## Pytorch setup

In [105]:
import torch

In [106]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load dataset

In [107]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

In [108]:
housing = fetch_california_housing()
x, y = housing.data, housing.target
feature_names = housing.feature_names
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_test, y_test, test_size=0.5, random_state=42)

In [109]:
print(f"Train set: {x_train.shape}, {y_train.shape}")
print(f"Test set: {x_test.shape}, {y_test.shape}")
print(f"Validation set: {x_val.shape}, {y_val.shape}")

Train set: (16512, 8), (16512,)
Test set: (2064, 8), (2064,)
Validation set: (2064, 8), (2064,)


## Preprocessing layer

In [ ]:
import numpy as np
import torch.nn as nn


class PreprocessingLayer(nn.Module):
    def __init__(self, feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5):
        super().__init__()
        self.feature_names = feature_names
        self.excluded_features = set(excluded_features)

        self.iqr_k = iqr_k

        excluded_idx = [i for i, name in enumerate(feature_names) if name in self.excluded_features]
        non_excluded_idx = [i for i, name in enumerate(feature_names) if name not in self.excluded_features]

        self.register_buffer("excluded_idx", torch.tensor(excluded_idx, dtype=torch.long))
        self.register_buffer("non_excluded_idx", torch.tensor(non_excluded_idx, dtype=torch.long))
        self.register_buffer("lower_bounds", torch.empty(len(non_excluded_idx)))
        self.register_buffer("upper_bounds", torch.empty(len(non_excluded_idx)))
        self.register_buffer("mean", torch.empty(len(non_excluded_idx)))
        self.register_buffer("std", torch.empty(len(non_excluded_idx)))

    def _to_tensor(self, x):
        if isinstance(x, np.ndarray):
            return torch.tensor(x, dtype=torch.float32)
        else:
            return x.float()

    def fit(self, x_train):
        x_train = self._to_tensor(x_train)

        # Get quantiles for non-excluded features to determine outlier bounds.
        x_non_excluded = x_train[:, self.non_excluded_idx]
        q1 = torch.quantile(x_non_excluded, 0.25, dim=0)
        q3 = torch.quantile(x_non_excluded, 0.75, dim=0)
        iqr = q3 - q1

        self.lower_bounds.copy_(q1 - self.iqr_k * iqr)
        self.upper_bounds.copy_(q3 + self.iqr_k * iqr)

        inlier_mask = ((x_non_excluded >= self.lower_bounds) & (x_non_excluded <= self.upper_bounds)).all(dim=1)
        x_inliers = x_train[inlier_mask]
        x_non_excluded_inliers = x_inliers[:, self.non_excluded_idx]

        self.mean.copy_(x_non_excluded_inliers.mean(dim=0))
        std = x_non_excluded_inliers.std(dim=0)

        # Avoid division by zero
        std = torch.where(std == 0, torch.ones_like(std), std)
        self.std.copy_(std)

        return self

    def transform(self, x, return_mask=False, remove_outliers=True):
        x = self._to_tensor(x)
        x_non_excluded = x[:, self.non_excluded_idx]
        inlier_mask = ((x_non_excluded >= self.lower_bounds) & (x_non_excluded <= self.upper_bounds)).all(dim=1)

        x_used = x[inlier_mask] if remove_outliers else x
        x_out = x_used.clone()

        # Standardize only non-excluded features for inliers.
        x_out[:, self.non_excluded_idx] = (x_out[:, self.non_excluded_idx] - self.mean) / self.std

        if return_mask:
            return x_out, inlier_mask
        return x_out

    def forward(self, x):
        # Keep batch size unchanged inside nn.Sequential.
        return self.transform(x, return_mask=False, remove_outliers=False)


preprocessor = PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5)
preprocessor.fit(x_train)

# Preprocess each split
x_train_pp, train_mask = preprocessor.transform(x_train, return_mask=True, remove_outliers=True)
x_val_pp, val_mask = preprocessor.transform(x_val, return_mask=True, remove_outliers=True)
x_test_pp, test_mask = preprocessor.transform(x_test, return_mask=True, remove_outliers=True)

y_train_pp = torch.tensor(y_train, dtype=torch.float32)[train_mask]
y_val_pp = torch.tensor(y_val, dtype=torch.float32)[val_mask]
y_test_pp = torch.tensor(y_test, dtype=torch.float32)[test_mask]

train_data = list(zip(x_train_pp, y_train_pp))
val_data = list(zip(x_val_pp, y_val_pp))
test_data = list(zip(x_test_pp, y_test_pp))

print(f"Train tuples: {len(train_data)}")
print(f"Val tuples:   {len(val_data)}")
print(f"Test tuples:  {len(test_data)}")

Train tuples: 13460
Val tuples:   1698
Test tuples:  1702


## Defining the models

In [111]:
import torch.optim as optim

input_size = len(feature_names)

# Model 1
model1 = nn.Sequential(
    PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5),
    nn.Linear(input_size, 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1)
)
model1[0].fit(x_train)
optimizer1 = optim.Adam(model1.parameters(), lr=0.001)

# Model 2
model2 = nn.Sequential(
    PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5),
    nn.Linear(input_size, 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1)
)
model2[0].fit(x_train)
optimizer2 = optim.Adam(model2.parameters(), lr=0.01)

# Model 3
model3 = nn.Sequential(
    PreprocessingLayer(feature_names=feature_names, excluded_features=("Latitude", "Longitude"), iqr_k=1.5),
    nn.Linear(input_size, 50),
    nn.ReLU(),
    nn.Linear(50, 25),
    nn.ReLU(),
    nn.Linear(25, 1)
)
model3[0].fit(x_train)
optimizer3 = optim.SGD(model3.parameters(), lr=0.001)

# Loss function for all 3 models
criterion = nn.MSELoss()

## Defining the minibatches creation function

In [112]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        # Targets should be float tensors with shape (batch_size, 1).
        y_batch = torch.stack([y[j] for j in batch_idx]).float().unsqueeze(1)
        yield x_batch, y_batch

## Defining the train function

In [113]:
from tqdm import tqdm

In [114]:
def train(model, train_data, val_data, optimizer, criterion, epochs, batch_size, patience=None, delta=0.005):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, _ in val_data]
    y_val = [y for _, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            # The dataset is using cpu, check if we are using gpu and move to gpu
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)
        # Eval
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, x_val, y_val):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

        # Early stopping
        if patience is not None:
            if epoch == 0:
                best_val_loss = epoch_val_loss
                patience_counter = 0
            else:
                if epoch_val_loss < best_val_loss - delta:
                    best_val_loss = epoch_val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1

            if patience_counter >= patience:
                print(f"Early stop triggered")
                break

    return epoch_train_loss, epoch_val_loss

## Trainning the models

In [115]:
model1_trained = train(model1, train_data, val_data, optimizer1, criterion, epochs=30, batch_size=64, patience=5)

Epoch 1/30: 100%|██████████| 211/211 [00:00<00:00, 284.42it/s]


Epoch [1/30], Loss: 1.3608, Val Loss: 1.0321


Epoch 2/30: 100%|██████████| 211/211 [00:00<00:00, 362.81it/s]


Epoch [2/30], Loss: 1.0213, Val Loss: 0.9104


Epoch 3/30: 100%|██████████| 211/211 [00:00<00:00, 368.36it/s]


...

Epoch [17/30], Loss: 0.4932, Val Loss: 0.4983


Epoch 18/30: 100%|██████████| 211/211 [00:00<00:00, 360.67it/s]


Epoch [18/30], Loss: 0.4895, Val Loss: 0.5055


Epoch 19/30: 100%|██████████| 211/211 [00:00<00:00, 378.06it/s]

Epoch [19/30], Loss: 0.4873, Val Loss: 0.5134
Early stop triggered
